<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# 5.3-B — COVERAGE vs LABEL PRECISION
# FAST COLAB VERSION
#
# Fixed MC budget per allocation:
#       R * m = 50,000
#
# Allocations:
#       (200,250)
#       (500,100)
#       (1000,50)
#       (2000,25)
#       (5000,10)
#
# Speed:
#   - parallel CPU exact sparse-LU targets
#   - exact target caching
#   - Numba-parallel nested Monte Carlo
#   - vectorized CUDA training
#   - mixed precision MLP on T4/L4
#   - model caching
#
# Same hazard architecture:
#       6 -> 128 -> 128 -> 128 -> 1
#       SiLU + sigmoid
#
# Same optimization:
#       AdamW
#       lr = 1e-3
#       weight decay = 1e-6
#       batch = 64
#       max epochs = 500
# ================================================================


# ================================================================
# 0. CPU THREAD SETTINGS
# ================================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# ================================================================
# 1. IMPORTS
# ================================================================

import time
import math
import random
import pickle

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

from numba import njit, prange, set_num_threads

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# ================================================================
# 2. CONFIG
# ================================================================

@dataclass
class C:

    seed:int = 20260820

    beta:Tuple[float,float] = (.30, 1.50)
    gamma:Tuple[float,float] = (.20, 1.00)
    omega:Tuple[float,float] = (.02, .50)
    frac:Tuple[float,float] = (.02, .20)

    trainN:Tuple[int,...] = tuple(
        range(40, 401, 20)
    )

    width:int = 128
    depth:int = 3

    batch:int = 64
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    patience:int = 20
    delta:float = 1e-6
    clip:float = 5.


cfg = C()


# ================================================================
# 3. FIXED MONTE CARLO BUDGET
# ================================================================

ALLOC = (
    (200, 250),
    (500, 100),
    (1000, 50),
    (2000, 25),
    (5000, 10)
)

BUDGET = 50_000

assert all(
    R * m == BUDGET
    for R, m in ALLOC
)

RMAX = max(
    R for R, m in ALLOC
)

Nscale = max(
    cfg.trainN
)

TEST_N = tuple(
    range(40, 401, 10)
)


# ================================================================
# 4. DIRECTORIES
# ================================================================

out = Path(
    "/content/results_5_3B_budget50000"
)

cache = Path(
    "/content/cache_5_3B_budget50000"
)

out.mkdir(
    parents=True,
    exist_ok=True
)

cache.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 5. HARDWARE
# ================================================================

CPU = (
    os.cpu_count()
    or 1
)

# LU is memory-intensive
N_EXACT = max(
    1,
    min(
        2,
        CPU
    )
)

# Numba MC can use visible CPU threads
N_MC = max(
    1,
    CPU
)

set_num_threads(
    N_MC
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = torch.cuda.is_available()

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "WARNING: CUDA unavailable."
    )

print(
    "CPU cores:",
    CPU
)

print(
    "Exact LU threads:",
    N_EXACT
)

print(
    "Numba MC threads:",
    N_MC
)

print(
    "Allocations:",
    ALLOC
)

print(
    "Budget per allocation:",
    f"{BUDGET:,}"
)


# ================================================================
# 6. REPRODUCIBILITY
# ================================================================

def seed_all(s):

    random.seed(s)

    np.random.seed(s)

    torch.manual_seed(s)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(s)


seed_all(
    cfg.seed
)


# ================================================================
# 7. RECORD
# ================================================================

@dataclass
class Rec:

    b:float
    g:float
    w:float

    N:int
    i0:int

    p:np.ndarray


# ================================================================
# 8. EXACT SIRS TOPOLOGY
# ================================================================

@lru_cache(None)
def topo(N):

    st = [
        (s, i)
        for i in range(1, N + 1)
        for s in range(N - i + 1)
    ]

    ix = {
        x:j
        for j, x in enumerate(st)
    }

    M = len(st)

    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]

    db=np.zeros(M)
    dg=np.zeros(M)
    dw=np.zeros(M)
    qb=np.zeros(M)

    for j, (s, i) in enumerate(st):

        r = N - s - i

        if s:

            ir.append(j)

            ic.append(
                ix[
                    (s - 1, i + 1)
                ]
            )

            ib.append(
                s * i / N
            )

            db[j] = (
                s * i / N
            )

        dg[j] = i

        if i == 1:

            qb[j] = i

        else:

            rr.append(j)

            rc.append(
                ix[
                    (s, i - 1)
                ]
            )

            rb.append(i)

        if r:

            wr.append(j)

            wc.append(
                ix[
                    (s + 1, i)
                ]
            )

            wb.append(r)

            dw[j] = r

    A = lambda x, d=float: np.asarray(
        x,
        dtype=d
    )

    return (
        ix,
        M,

        A(ir, int),
        A(ic, int),
        A(ib),

        A(rr, int),
        A(rc, int),
        A(rb),

        A(wr, int),
        A(wc, int),
        A(wb),

        db,
        dg,
        dw,
        qb
    )


# ================================================================
# 9. EXACT PMF
# ================================================================

def exact_p(
    b,
    g,
    w,
    N,
    i0
):

    (
        ix,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topo(N)

    rows = np.r_[
        ir,
        rr,
        wr,
        np.arange(M)
    ]

    cols = np.r_[
        ic,
        rc,
        wc,
        np.arange(M)
    ]

    vals = np.r_[
        b * ib,
        g * rb,
        w * wb,
        -(b * db + g * dg + w * dw)
    ]

    T = sparse.coo_matrix(
        (
            vals,
            (rows, cols)
        ),
        shape=(M, M)
    ).tocsc()

    D1 = sparse.coo_matrix(
        (
            b * ib,
            (ir, ic)
        ),
        shape=(M, M)
    ).tocsc()

    A = (
        -(
            T - D1
        )
    ).tocsc()

    lu = splu(A)

    q = (
        g * qb
    )

    alpha = np.zeros(M)

    alpha[
        ix[
            (N - i0, i0)
        ]
    ] = 1.

    bb = lu.solve(q)

    v = alpha.copy()

    p = np.zeros(
        N + 2
    )

    for k in range(
        N + 1
    ):

        p[k] = (
            v @ bb
        )

        v = np.asarray(
            D1.T
            @
            lu.solve(
                v,
                trans="T"
            )
        ).ravel()

    p[-1] = (
        v.sum()
    )

    p[
        np.abs(p) < 1e-12
    ] = 0.

    p = np.maximum(
        p,
        0.
    )

    mass = p.sum()

    if (
        not np.isfinite(mass)
        or
        mass <= 0
    ):

        raise RuntimeError(
            "Invalid exact PMF."
        )

    return (
        p / mass
    )


# ================================================================
# 10. DESIGN
# ================================================================

def design(
    n,
    Ns,
    s
):

    U = qmc.LatinHypercube(
        4,
        seed=s
    ).random(n)

    scale = lambda x,a: (
        a[0]
        +
        (a[1] - a[0]) * x
    )

    b = scale(
        U[:,0],
        cfg.beta
    )

    g = scale(
        U[:,1],
        cfg.gamma
    )

    w = scale(
        U[:,2],
        cfg.omega
    )

    f = scale(
        U[:,3],
        cfg.frac
    )

    Nv = np.tile(
        Ns,
        math.ceil(
            n / len(Ns)
        )
    )[:n]

    rng = np.random.default_rng(
        s + 99
    )

    rng.shuffle(Nv)

    i0 = np.array([
        int(
            np.clip(
                round(
                    f[j] * Nv[j]
                ),
                2,
                Nv[j]
            )
        )
        for j in range(n)
    ])

    for N in Ns:

        z = np.where(
            Nv == N
        )[0]

        i0[
            rng.choice(
                z,
                max(
                    1,
                    round(
                        .25 * len(z)
                    )
                ),
                False
            )
        ] = 1

    return [
        (
            float(b[j]),
            float(g[j]),
            float(w[j]),
            int(Nv[j]),
            int(i0[j])
        )
        for j in range(n)
    ]


# ================================================================
# 11. PARALLEL EXACT TARGETS + CACHE
# ================================================================

def _exact_one(
    j,
    z
):

    return (
        j,
        Rec(
            *z,
            exact_p(*z)
        )
    )


def exact_set(
    configs,
    name,
    file
):

    file = Path(file)

    if file.exists():

        with open(
            file,
            "rb"
        ) as f:

            x = pickle.load(f)

        print(
            f"{name}: cache loaded ({len(x)})"
        )

        return x

    jobs = list(
        enumerate(configs)
    )

    jobs.sort(
        key=lambda x:
        x[1][3],
        reverse=True
    )

    print(
        f"\n{name}: "
        f"{len(jobs)} exact PMFs | "
        f"{N_EXACT} threads"
    )

    t0 = time.perf_counter()

    z = Parallel(
        n_jobs=N_EXACT,
        backend="threading",
        verbose=5
    )(
        delayed(
            _exact_one
        )(
            j,
            x
        )
        for j,x in jobs
    )

    z.sort(
        key=lambda x:
        x[0]
    )

    ans = [
        r
        for _,r in z
    ]

    with open(
        file,
        "wb"
    ) as f:

        pickle.dump(
            ans,
            f,
            pickle.HIGHEST_PROTOCOL
        )

    print(
        f"{name}: "
        f"{time.perf_counter()-t0:.1f}s"
    )

    return ans


# ================================================================
# 12. NUMBA MONTE CARLO
# ================================================================

@njit
def sim_C_numba(
    b,
    g,
    w,
    N,
    i0
):

    S = N - i0
    I = i0
    R = 0
    C = 0

    while I > 0:

        infection = (
            b * S * I / N
        )

        recovery = (
            g * I
        )

        waning = (
            w * R
        )

        z = (
            np.random.random()
            *
            (
                infection
                +
                recovery
                +
                waning
            )
        )

        if z < infection:

            S -= 1
            I += 1
            C += 1

        elif z < (
            infection
            +
            recovery
        ):

            I -= 1
            R += 1

        else:

            R -= 1
            S += 1

    return min(
        C,
        N + 1
    )


# ================================================================
# 13. NESTED MONTE CARLO
#
# Required levels:
#     10, 25, 50, 100, 250
#
# max simulations/config:
#
#     j < 200       : 250
#     200-499       : 100
#     500-999       : 50
#     1000-1999     : 25
#     2000-4999     : 10
#
# Actual total:
#
# 200*250
# +300*100
# +500*50
# +1000*25
# +3000*10
# = 160,000 epidemics
# ================================================================

@njit(parallel=True)
def nested_mc_counts(
    B,
    G,
    W,
    N,
    I0,
    maxm,
    maxN,
    seed
):

    R = len(N)

    levels = np.array(
        [10,25,50,100,250]
    )

    snap = np.zeros(
        (
            5,
            R,
            maxN + 2
        ),
        dtype=np.int32
    )

    for j in prange(R):

        np.random.seed(
            seed
            +
            100003 * j
        )

        h = np.zeros(
            maxN + 2,
            dtype=np.int32
        )

        level = 0

        for q in range(
            1,
            maxm[j] + 1
        ):

            c = sim_C_numba(
                B[j],
                G[j],
                W[j],
                N[j],
                I0[j]
            )

            h[c] += 1

            while (
                level < 5
                and
                q == levels[level]
            ):

                snap[
                    level,
                    j,
                    :
                ] = h

                level += 1

    return snap


def make_nested_mc(
    records,
    file
):

    file = Path(file)

    if file.exists():

        print(
            "Nested MC cache loaded."
        )

        with open(
            file,
            "rb"
        ) as f:

            return pickle.load(f)

    B = np.asarray(
        [r.b for r in records],
        np.float64
    )

    G = np.asarray(
        [r.g for r in records],
        np.float64
    )

    W = np.asarray(
        [r.w for r in records],
        np.float64
    )

    N = np.asarray(
        [r.N for r in records],
        np.int64
    )

    I0 = np.asarray(
        [r.i0 for r in records],
        np.int64
    )

    maxm = np.zeros(
        len(records),
        dtype=np.int64
    )

    maxm[:200] = 250
    maxm[200:500] = 100
    maxm[500:1000] = 50
    maxm[1000:2000] = 25
    maxm[2000:5000] = 10

    print(
        "\nNested Monte Carlo generation"
    )

    print(
        "Actual simulated epidemics:",
        f"{maxm.sum():,}"
    )

    # First call also includes Numba compilation.
    t0 = time.perf_counter()

    snap = nested_mc_counts(
        B,
        G,
        W,
        N,
        I0,
        maxm,
        max(cfg.trainN),
        cfg.seed + 999
    )

    print(
        f"Nested MC time: "
        f"{time.perf_counter()-t0:.1f}s"
    )

    level_index = {
        10:0,
        25:1,
        50:2,
        100:3,
        250:4
    }

    labels = {}

    for R,m in ALLOC:

        level = level_index[m]

        labels[
            (R,m)
        ] = [
            snap[
                level,
                j,
                :records[j].N + 2
            ].astype(
                np.float64
            ) / m

            for j in range(R)
        ]

    with open(
        file,
        "wb"
    ) as f:

        pickle.dump(
            labels,
            f,
            pickle.HIGHEST_PROTOCOL
        )

    del snap

    return labels


# ================================================================
# 14. HAZARD NETWORK
# ================================================================

class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        layers = []

        d = 6

        for _ in range(
            cfg.depth
        ):

            layers += [
                nn.Linear(
                    d,
                    cfg.width
                ),
                nn.SiLU()
            ]

            d = cfg.width

        layers.append(
            nn.Linear(
                d,
                1
            )
        )

        self.net = nn.Sequential(
            *layers
        )

    def forward(
        self,
        x
    ):

        return torch.sigmoid(
            self.net(x).squeeze(-1)
        )


# ================================================================
# 15. PACK DATA TO GPU
# ================================================================

def pack(
    records,
    labels=None
):

    groups = {}

    for j,r in enumerate(records):

        groups.setdefault(
            r.N,
            []
        ).append(j)

    P = {}

    for N,idx in groups.items():

        idx = np.asarray(
            idx
        )

        B = len(idx)
        K = N + 1

        beta = torch.tensor(
            [records[j].b for j in idx],
            dtype=torch.float32,
            device=device
        )[:,None]

        gamma = torch.tensor(
            [records[j].g for j in idx],
            dtype=torch.float32,
            device=device
        )[:,None]

        omega = torch.tensor(
            [records[j].w for j in idx],
            dtype=torch.float32,
            device=device
        )[:,None]

        ns = torch.full(
            (B,1),
            N / Nscale,
            dtype=torch.float32,
            device=device
        )

        i0 = torch.tensor(
            [
                records[j].i0 / N
                for j in idx
            ],
            dtype=torch.float32,
            device=device
        )[:,None]

        c = (
            torch.arange(
                K,
                dtype=torch.float32,
                device=device
            )
            /
            N
        )[None,:]

        X = torch.stack(
            [
                beta.expand(B,K),
                gamma.expand(B,K),
                omega.expand(B,K),
                ns.expand(B,K),
                i0.expand(B,K),
                c.expand(B,K)
            ],
            dim=2
        )

        Y = np.stack([
            records[j].p
            if labels is None
            else labels[j]
            for j in idx
        ])

        P[N] = {
            "X":X,
            "Y":torch.tensor(
                Y,
                dtype=torch.float32,
                device=device
            ),
            "n":B
        }

    return P


# ================================================================
# 16. PMF + TAIL
# ================================================================

def pmf_from_h(h):

    # keep reconstruction in float32
    h = h.float()

    B = h.shape[0]

    survival = torch.cat(
        [
            torch.ones(
                (B,1),
                dtype=torch.float32,
                device=h.device
            ),

            torch.cumprod(
                1 - h[:,:-1],
                dim=1
            )
        ],
        dim=1
    )

    return torch.cat(
        [
            survival * h,

            torch.prod(
                1 - h,
                dim=1,
                keepdim=True
            )
        ],
        dim=1
    )


def tail(P):

    return torch.flip(
        torch.cumsum(
            torch.flip(
                P[:,1:],
                dims=[1]
            ),
            dim=1
        ),
        dims=[1]
    )


# ================================================================
# 17. GPU FORWARD
# ================================================================

def predict_batch(
    net,
    X
):

    B,K,_ = X.shape

    if USE_AMP:

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            h = net(
                X.reshape(
                    B*K,
                    6
                )
            ).reshape(
                B,
                K
            )

    else:

        h = net(
            X.reshape(
                B*K,
                6
            )
        ).reshape(
            B,
            K
        )

    return pmf_from_h(h)


def batch_loss(
    net,
    X,
    Y
):

    P = predict_batch(
        net,
        X
    )

    Lp = torch.sum(
        (P - Y)**2,
        dim=1
    )

    Lrho = torch.mean(
        (
            tail(P)
            -
            tail(Y)
        )**2,
        dim=1
    )

    return (
        Lp + Lrho
    ).mean()


# ================================================================
# 18. MINI-BATCH SCHEDULE
# ================================================================

def schedule(
    P,
    rng
):

    ans = []

    for N,G in P.items():

        z = rng.permutation(
            G["n"]
        )

        ans.extend([
            (
                N,
                z[
                    s:s + cfg.batch
                ]
            )
            for s in range(
                0,
                len(z),
                cfg.batch
            )
        ])

    rng.shuffle(ans)

    return ans


# ================================================================
# 19. VALIDATION
# ================================================================

@torch.no_grad()
def val_loss(
    net,
    P
):

    net.eval()

    total = 0.
    n = 0

    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X = G["X"][
                s:s + cfg.batch
            ]

            Y = G["Y"][
                s:s + cfg.batch
            ]

            L = batch_loss(
                net,
                X,
                Y
            )

            total += (
                L.item()
                *
                len(X)
            )

            n += len(X)

    return (
        total / n
    )


# ================================================================
# 20. TRAINING + MODEL CACHE
# ================================================================

def fit(
    TR,
    VA,
    seed,
    model_file
):

    model_file = Path(
        model_file
    )

    net = HazardNet().to(
        device
    )

    if model_file.exists():

        ck = torch.load(
            model_file,
            map_location=device
        )

        net.load_state_dict(
            ck["state"]
        )

        print(
            "Model cache loaded | "
            f"best epoch={ck['epoch']}"
        )

        return (
            net,
            0.
        )

    seed_all(seed)

    opt = torch.optim.AdamW(
        net.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.wd
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt,
            factor=.5,
            patience=15
        )
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    rng = np.random.default_rng(
        seed + 77
    )

    best = np.inf
    best_epoch = 0
    state = None
    wait = 0

    t0 = time.perf_counter()

    for epoch in range(
        1,
        cfg.epochs + 1
    ):

        net.train()

        for N,idx in schedule(
            TR,
            rng
        ):

            X = TR[N]["X"][idx]
            Y = TR[N]["Y"][idx]

            opt.zero_grad(
                set_to_none=True
            )

            # predict_batch handles AMP
            L = batch_loss(
                net,
                X,
                Y
            )

            scaler.scale(
                L
            ).backward()

            scaler.unscale_(
                opt
            )

            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                cfg.clip
            )

            scaler.step(
                opt
            )

            scaler.update()

        v = val_loss(
            net,
            VA
        )

        scheduler.step(v)

        if (
            state is None
            or
            v < best - cfg.delta
        ):

            best = v
            best_epoch = epoch
            wait = 0

            state = {
                k:
                x.detach().cpu().clone()
                for k,x
                in net.state_dict().items()
            }

        else:

            wait += 1

        if (
            epoch == 1
            or
            epoch % 20 == 0
        ):

            print(
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )

        if wait >= cfg.patience:

            print(
                "Early stopping:",
                epoch
            )

            break

    net.load_state_dict(
        state
    )

    sec = (
        time.perf_counter()
        -
        t0
    )

    torch.save(
        {
            "state":state,
            "epoch":best_epoch,
            "validation":best
        },
        model_file
    )

    print(
        f"fit complete | "
        f"best epoch={best_epoch} | "
        f"time={sec:.1f}s"
    )

    return (
        net,
        sec
    )


# ================================================================
# 21. TEST METRICS
# ================================================================

@torch.no_grad()
def metrics(
    net,
    P
):

    net.eval()

    E2 = []
    Erho = []

    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X = G["X"][
                s:s + cfg.batch
            ]

            Y = G["Y"][
                s:s + cfg.batch
            ]

            Ph = predict_batch(
                net,
                X
            )

            E2.append(
                torch.linalg.vector_norm(
                    Ph - Y,
                    dim=1
                )
                .cpu()
                .numpy()
            )

            Erho.append(
                torch.max(
                    torch.abs(
                        tail(Ph)
                        -
                        tail(Y)
                    ),
                    dim=1
                )
                .values
                .cpu()
                .numpy()
            )

    return (
        np.concatenate(E2),
        np.concatenate(Erho)
    )


# ================================================================
# 22. GENERATE DESIGNS
# ================================================================

print(
    "\nGenerating designs..."
)

train_design = design(
    RMAX,
    cfg.trainN,
    cfg.seed + 1
)

val_design = design(
    400,
    cfg.trainN,
    cfg.seed + 2
)

test_design = design(
    700,
    TEST_N,
    cfg.seed + 3
)


# ================================================================
# 23. EXACT DATASETS
# ================================================================

train = exact_set(
    train_design,
    "TRAIN",
    cache / "exact_train_R5000.pkl"
)

val = exact_set(
    val_design,
    "VAL",
    cache / "exact_val_400.pkl"
)

test = exact_set(
    test_design,
    "TEST",
    cache / "exact_test_700.pkl"
)


# ================================================================
# 24. PACK VALIDATION + TEST
# ================================================================

VA = pack(
    val
)

TE = pack(
    test
)


# ================================================================
# 25. EXACT TEACHER REFERENCE
# ================================================================

print(
    "\n"
    +
    "="*70
)

print(
    "EXACT TEACHER | R=5000"
)

print(
    "="*70
)

TR = pack(
    train
)

exact_net, exact_train_sec = fit(
    TR,
    VA,
    cfg.seed + 7001,
    cache / "model_exact_R5000.pt"
)

ee, rr = metrics(
    exact_net,
    TE
)

exact_E2 = np.median(
    ee
)

exact_Erho = np.median(
    rr
)

exact_E2_p95 = np.quantile(
    ee,
    .95
)

exact_Erho_p95 = np.quantile(
    rr,
    .95
)

print(
    f"Exact E2: "
    f"median={exact_E2:.6g}, "
    f"p95={exact_E2_p95:.6g}"
)

print(
    f"Exact Erho: "
    f"median={exact_Erho:.6g}, "
    f"p95={exact_Erho_p95:.6g}"
)

del TR
del exact_net

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ================================================================
# 26. GENERATE ALL MC LABELS ONCE
# ================================================================

labels_all = make_nested_mc(
    train,
    cache / "nested_MC_budget50000.pkl"
)


# ================================================================
# 27. FIXED-BUDGET MC EXPERIMENT
# ================================================================

rows = []

for R,m in ALLOC:

    print(
        "\n"
        +
        "="*70
    )

    print(
        f"MC TEACHER | "
        f"R={R}, "
        f"m={m}, "
        f"Rm={R*m:,}"
    )

    print(
        "="*70
    )

    labels = labels_all[
        (R,m)
    ]

    TR = pack(
        train[:R],
        labels
    )

    net, training_sec = fit(
        TR,
        VA,
        cfg.seed + 7001,
        cache / f"model_MC_R{R}_m{m}.pt"
    )

    e2, erho = metrics(
        net,
        TE
    )

    rows.append(
        [
            R,
            m,
            R*m,

            training_sec,

            np.median(e2),
            np.quantile(e2,.25),
            np.quantile(e2,.75),
            np.quantile(e2,.95),

            np.median(erho),
            np.quantile(erho,.25),
            np.quantile(erho,.75),
            np.quantile(erho,.95)
        ]
    )

    print(
        f"E2: "
        f"median={np.median(e2):.6g}, "
        f"p95={np.quantile(e2,.95):.6g}"
    )

    print(
        f"Erho: "
        f"median={np.median(erho):.6g}, "
        f"p95={np.quantile(erho,.95):.6g}"
    )

    del TR
    del net

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ================================================================
# 28. TABLE
# ================================================================

columns = [
    "R",
    "m",
    "R_times_m",

    "training_sec",

    "E2_med",
    "E2_q1",
    "E2_q3",
    "E2_p95",

    "Erho_med",
    "Erho_q1",
    "Erho_q3",
    "Erho_p95"
]

df = pd.DataFrame(
    rows,
    columns=columns
)

df.to_csv(
    out / "coverage_vs_precision.csv",
    index=False
)

(
    out / "coverage_vs_precision.tex"
).write_text(
    df.to_latex(
        index=False,
        float_format="%.4g"
    )
)

print(
    "\nFINAL RESULTS\n"
)

print(
    df.to_string(
        index=False
    )
)


# ================================================================
# 29. MAIN FIGURE — MEDIAN + IQR
# ================================================================

fig, ax = plt.subplots(
    1,
    2,
    figsize=(10,4)
)

for a,y,q1,q3,base,title in [

    (
        ax[0],
        "E2_med",
        "E2_q1",
        "E2_q3",
        exact_E2,
        r"$E_2$"
    ),

    (
        ax[1],
        "Erho_med",
        "Erho_q1",
        "Erho_q3",
        exact_Erho,
        r"$E_\rho$"
    )

]:

    x = df["R"].to_numpy()

    a.plot(
        x,
        df[y],
        "o-",
        lw=2,
        color="#0072B2",
        label=r"MC, fixed $Rm=50{,}000$"
    )

    a.fill_between(
        x,
        df[q1].to_numpy(),
        df[q3].to_numpy(),
        color="#0072B2",
        alpha=.15
    )

    a.axhline(
        base,
        color="#D55E00",
        ls="--",
        lw=2,
        label=r"Exact teacher, $R=5000$"
    )

    a.set_xscale(
        "log"
    )

    a.set_xticks(
        x
    )

    a.set_xticklabels(
        x.astype(int)
    )

    a.set_xlabel(
        r"Training configurations $R$"
    )

    a.set_ylabel(
        "Median test error"
    )

    a.set_title(
        title
    )

    a.grid(
        alpha=.15
    )

    a.legend(
        frameon=False,
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    out / "coverage_vs_precision.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 30. P95 FIGURE
# ================================================================

fig, ax = plt.subplots(
    1,
    2,
    figsize=(10,4)
)

for a,y,base,title in [

    (
        ax[0],
        "E2_p95",
        exact_E2_p95,
        r"$E_2$"
    ),

    (
        ax[1],
        "Erho_p95",
        exact_Erho_p95,
        r"$E_\rho$"
    )

]:

    x = df["R"].to_numpy()

    a.plot(
        x,
        df[y],
        "o-",
        lw=2,
        color="#0072B2",
        label=r"MC, fixed $Rm=50{,}000$"
    )

    a.axhline(
        base,
        color="#D55E00",
        ls="--",
        lw=2,
        label=r"Exact teacher, $R=5000$"
    )

    a.set_xscale(
        "log"
    )

    a.set_xticks(
        x
    )

    a.set_xticklabels(
        x.astype(int)
    )

    a.set_xlabel(
        r"Training configurations $R$"
    )

    a.set_ylabel(
        "95th percentile test error"
    )

    a.set_title(
        title
    )

    a.grid(
        alpha=.15
    )

    a.legend(
        frameon=False,
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    out / "coverage_vs_precision_p95.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 31. TRAINING TIME FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(5.5,4)
)

ax.plot(
    df["R"],
    df["training_sec"],
    "o-",
    lw=2,
    color="#CC79A7"
)

ax.set_xscale(
    "log"
)

ax.set_xticks(
    df["R"]
)

ax.set_xticklabels(
    df["R"].astype(int)
)

ax.set_xlabel(
    r"Training configurations $R$"
)

ax.set_ylabel(
    "Neural training time (seconds)"
)

ax.set_title(
    "GPU training time"
)

ax.grid(
    alpha=.15
)

plt.tight_layout()

plt.savefig(
    out / "coverage_vs_precision_training_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 32. FINAL
# ================================================================

print(
    "\n"
    +
    "="*70
)

print(
    "DONE"
)

print(
    "="*70
)

print(
    "Fixed budget per allocation:",
    f"{BUDGET:,}"
)

print(
    "Nominal MC uses across 5 allocations:",
    f"{BUDGET*len(ALLOC):,}"
)

print(
    "Actual epidemics simulated by nested construction:",
    f"{160_000:,}"
)

print(
    "Results:",
    out
)

print(
    "Cache:",
    cache
)

print(
    f"Exact reference: "
    f"E2={exact_E2:.6g}, "
    f"Erho={exact_Erho:.6g}"
)

CPU cores: 2
Exact LU threads: 2
Numba MC threads: 2
Allocations: ((200, 250), (500, 100), (1000, 50), (2000, 25), (5000, 10))
Budget per allocation: 50,000

Generating designs...

TRAIN: 5000 exact PMFs | 2 threads


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  1.2min
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:  5.7min
[Parallel(n_jobs=2)]: Done 158 tasks      | elapsed: 13.2min
[Parallel(n_jobs=2)]: Done 284 tasks      | elapsed: 23.4min
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed: 35.0min
[Parallel(n_jobs=2)]: Done 644 tasks      | elapsed: 48.0min
[Parallel(n_jobs=2)]: Done 878 tasks      | elapsed: 61.4min
[Parallel(n_jobs=2)]: Done 1148 tasks      | elapsed: 74.5min
[Parallel(n_jobs=2)]: Done 1454 tasks      | elapsed: 86.7min
[Parallel(n_jobs=2)]: Done 1796 tasks      | elapsed: 97.7min
[Parallel(n_jobs=2)]: Done 2174 tasks      | elapsed: 106.5min
[Parallel(n_jobs=2)]: Done 2588 tasks      | elapsed: 113.5min
[Parallel(n_jobs=2)]: Done 3038 tasks      | elapsed: 118.3min
[Parallel(n_jobs=2)]: Done 3524 tasks      | elapsed: 121.0min
[Parallel(n_jobs=2)]: Done 4046 tasks      | elapsed: 1

TRAIN: 7327.6s

VAL: 400 exact PMFs | 2 threads


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  1.2min
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:  4.9min
[Parallel(n_jobs=2)]: Done 158 tasks      | elapsed:  8.2min
[Parallel(n_jobs=2)]: Done 284 tasks      | elapsed:  9.7min
[Parallel(n_jobs=2)]: Done 400 out of 400 | elapsed:  9.7min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.


VAL: 584.1s

TEST: 700 exact PMFs | 2 threads


[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:  1.2min
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed:  5.2min
[Parallel(n_jobs=2)]: Done 158 tasks      | elapsed:  9.9min
[Parallel(n_jobs=2)]: Done 284 tasks      | elapsed: 13.9min
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed: 16.1min
[Parallel(n_jobs=2)]: Done 644 tasks      | elapsed: 16.5min
[Parallel(n_jobs=2)]: Done 700 out of 700 | elapsed: 16.5min finished


TEST: 990.8s

EXACT TEACHER | R=5000
epoch=  1 | val=1.9793e-01 | best=1.9793e-01 | wait=0
epoch= 20 | val=1.1793e-01 | best=1.1771e-01 | wait=3
epoch= 40 | val=5.1323e-02 | best=4.2223e-02 | wait=1
epoch= 60 | val=3.4222e-02 | best=2.3766e-02 | wait=1
epoch= 80 | val=1.4222e-02 | best=1.4222e-02 | wait=0
epoch=100 | val=1.7864e-02 | best=8.3759e-03 | wait=1
epoch=120 | val=2.0203e-02 | best=4.9300e-03 | wait=1
epoch=140 | val=5.3075e-03 | best=4.0025e-03 | wait=6
epoch=160 | val=3.2756e-03 | best=2.7432e-03 | wait=2
epoch=180 | val=2.2176e-03 | best=2.2176e-03 | wait=0
epoch=200 | val=2.3138e-03 | best=1.8874e-03 | wait=6
epoch=220 | val=3.4819e-03 | best=1.6652e-03 | wait=11
epoch=240 | val=2.3677e-03 | best=1.3724e-03 | wait=6
epoch=260 | val=2.0189e-03 | best=1.2945e-03 | wait=14
epoch=280 | val=1.1678e-03 | best=1.1678e-03 | wait=0
epoch=300 | val=3.0466e-03 | best=9.8000e-04 | wait=11
epoch=320 | val=2.4053e-03 | best=8.6110e-04 | wait=6
epoch=340 | val=7.9383e-04 | best=7.4487e-